# Environmental Analysis -- Practice Set 13: Combining sentiment analysis and topic modelling

In this week's lecture material, you learned about LDA and sentiment analysis techniques for climate tweets and speeches data. In this practice set, you will combine those techniques by conducting topic modelling on the climate speeches data, and generating a sentiment index for those topics. 

This can be useful for tracking the temperature of climate discourse over time - did climate discourse become more negative as more assessment reports were released, or more positive as technological innovation advanced? You can link this with other datasets to draw some kind of relationship between _how we talk_ about climate, and how that might affect _how we actually take action_ on climate?

Remember to include comments with meaningful descriptions for each step in your code.

As always, use sources like Google, Stack Overflow, documentation, and other forums for help. Try to avoid AI in order to develop your own ability to conceptualize solution strategies.

First, make sure required packages are installed by running the following code block. 

In [ ]:
# 1. Speeds up compilation by using all available CPU cores
options(Ncpus = parallel::detectCores())

# 2. Installs the critical system dependencies required by tm, tidyverse, and topicmodels
system("sudo apt-get update && sudo apt-get install -y libgsl-dev libxml2-dev libssl-dev libcurl4-openssl-dev")

# 3. Quick, automated bulk installation of the packages
install.packages(c(
  "tidyverse", "tidytext", "topicmodels",
  "SnowballC", "tm", "knitr", "scales",
  "RColorBrewer", "broom"
), repos = "https://r-project.org", quiet = TRUE)

# 4. Verification Check: Attempts to load every package
packages <- c("tidyverse", "tidytext", "topicmodels", "SnowballC", "tm",
              "ggplot2", "dplyr", "stringr", "knitr", "scales", "RColorBrewer", "broom")

verify_load <- sapply(packages, require, character.only = TRUE)

# 5. Print out the final success/failure summary
cat("\n--- INSTALLATION VERIFICATION STATUS ---\n")
print(verify_load)
if(all(verify_load)) {
  cat("\n All libraries installed and loaded successfully!\n")
} else {
  cat("\n WARNING: Some libraries failed to load. Check the list above.\n")
}

### Question 1: Warm-up with regex / stringr recap (3 points)

We learned about text cleaning data using `stringr` tools and regex (Regular Expression) in Course 3 Module 10. Let's quickly recap some of those tools again before diving into this week's practice set.

To recap, you may refer back to your code-along notebook from Course 3 Module 10. For additional resources:
  - RStudio has a cheat sheet of stringr functions [here](https://rstudio.github.io/cheatsheets/strings.pdf). The second page contains information about regex.
  - Read more about regex [here](https://en.wikipedia.org/wiki/Regular_expression) and [here](https://www.statsig.com/perspectives/what-is-regex-a-beginner%27s-guide-to-pattern-matching). 
  - [RegExr](https://regexr.com/) is a useful resource for understanding regex you might find out in the wild in code, or for building your own regex for matching a desired pattern.

1. Consider the sentence "The quick gray fox jumped over the lazy dog." (1 point)
   - Now, write code to find the word after "fox" using stringr or regex tools. 
2. Suppose you had a large quantity of standardized environmental reports that were produced across different towns every month. (2 points)
   - The report files are named in the format "TOWN NAME_mmyy.txt" (e.g. "redtown_0109.txt"). 
   - The reports all start with the same sentence, "This report was produced using methodology pwsID {xxxx}," where xxxx is a 4-digit ID number identifying the methodology used for the report.
   - You can assume that the string "pwsID" only appears once in each report.
   - How would you extract the pwsID number for a given report file name using stringr or regex tools? 
   - You may use the pseudocode fragment provided. Include the specific stringr functions and regex expressions you would use.

In [0]:
# ✏️ question 1
sentence <- "The quick gray fox jumped over the lazy dogs."
# use stringr and regex tools to output the word after fox
# we will award credit by verifying you have printed the correct output using the requested tools


# ✏️ question 2
# you may use the fragment provided below. 
# describe how you would extract the pwsID number for a given report file name.
# we will award credit by verifying you have demonstrated correct understanding using the requested tools
# to reiterate - you do not need to provide fully functional code to solve the problem
# pseudocode, along with a specific regex/stringr application, will suffice for full credit.

# extract_pwsID <- function(filename){
# ✏️ INSERT YOUR CODE HERE
#}
# lapply(ls(pattern='.txt'), extract_pwsID) # applies function to all .txt files in directory

## Question 2: Tokenizing and Cleaning Text Data (7 points)

1. Load the relevant libraries for working with spatial data 
2. Load the `climate_speeches` datase
3. Identify the top 20 countries by number of speeches (2 points)
    - Use the `climate_speeches$country` column to help you do this. Store the vector of the desired country codes in an object called `top_countries` 
    - To get the country names from the country codes, load the `countrycode` library and run `countrycode(top_countries, 'iso3c', 'country.name')`
    - What do you notice about these countries? What do they have in common? 
    - 1 of these countries is the odd one out based on this commonality. Store that country's 3-letter code in an object called `odd_country`. 
4. Filter the `climate_speeches` dataset so it only contains the 19 countries we just identified. Store the new dataset in an object called `top_climate_speeches` (1 point)
5. Tokenize the filtered `climate_speeches`. Store it an object called `raw_climate_tokens` (1 point)
6. Clean the tokens. For each step, store the output in a new object for auto-grading (3 points)
    - Remove stop words: store in `climate_tokens_1`
    - Remove numbers (do this as its own step!): store in `climate_tokens_2`
    - Remove single letters (do this as its own step!): store in `climate_tokens_3`
        - Hint: In the code-along notebook, we ran: `filter(!str_detect(word,"^\\d+$|^[a-z]$"))`. What does the expression inside the `filter` function mean? 
        - What about the expression `"^\\d+$|^[a-z]$"` specifically?
        - Ultimately, these two steps should require you to make only one minor modification to the original code.
    - Stem words: store in `climate_tokens_clean`

In [ ]:
# Load the required packages
library(tidyverse)     # Data manipulation and visualization
library(tidytext)      # Text mining with tidy data principles
library(topicmodels)   # Topic modeling (LDA)
library(SnowballC)     # Stemming
library(tm)            # Text mining utilities
library(ggplot2)       # Data visualization
library(dplyr)         # Data manipulation
library(stringr)       # String manipulation
library(knitr)         # For nice table output
library(RColorBrewer)  # Color palettes
library(broom) 
library(countrycode)

In [ ]:
# Load the climate speeches dataset
climate_data <- read_csv("https://raw.githubusercontent.com/yse-eds-cert/practice-set-13/main/data/climate_data_course.csv")

# Load the AFINN sentiment lexicon (provided as a .csv so you don't need to run get_sentiments("afinn"))
afinn <- read_csv("https://raw.githubusercontent.com/yse-eds-cert/practice-set-13/main/data/afinn.csv") %>% select(word, value)



In [0]:
# ✏️ Get top 20 countries

In [0]:
# ✏️ Filter the data to the top 20 countries

In [0]:
#✏️ Tokenize
# to-do: break down the tokenizing code into steps (and remove this comment)

## Question 3: Sentiment Analysis (8 points)

1. Calculate the net sentiment (positive minus negative) associated with each speech. Use 2 different lexicons: Bing and Afinn (2 points)
   - Store your results in a data-frame called `sentiment_df` with 5 columns: `filename`, `country`, `year`, `bing`, and `afinn`  
   - Read more about different lexicons [here](https://bookdown.org/psonkin18/berkshire/sentiment.html) and [here](https://books.psychstat.org/textmining/sentiment-analysis.html).
   - We have provided the afinn lexicon directly as a .csv file for you to use. You can download it yourself on your local device by running `get_sentiments('afinn')` on RStudio and following the prompts.
2. Start with the Bing lexicon. Plot the mean sentiment of speeches over time (use `ggplot` and store the plot in an object called `overall_bing_sentiment_plot`). (1 point)
    - Is it increasing or decreasing? (We will not grade you for this.)
3. Plot the mean sentiment of speeches **by country** over time. Which country saw the biggest drop in positive sentiment over the time period? (I.e. saw the biggest movement towards negative sentiment in their speeches.) Store its 3-letter code in an object called `pessimistic_bing_country` (2 points)
	- We are defining "change in sentiment over the time period" as the **difference in** the country's **mean sentiments** of speeches in the **first and last years** it appears in the dataset, notwithstanding the fact that countries first and last appear in the dataset different years.
   - For example, if country A first appears in `top_climate_speeches` in 1991 and last appears in 2023, its change in sentiment is the mean sentiment of its speeches in 2023, minus the mean sentiment of its speeches in 1991. If country B first appears in 1992 and last appears in 2022, its change in sentiment is calculated similarly, and we compare the two countries' changes in sentiment directly against each other, even though they first and last appear in `top_climate_speeches` at different times.
4. Repeat steps 2 and 3 with the Afinn lexicon. (3 points)
   - Plot the mean sentiment of speeches over time (use `ggplot` and store the plot in an object called `overall_afinn_sentiment_plot`). 
   - Is it increasing or decreasing? (We will not grade you for this.)
   - Which country saw the biggest drop in sentiment over the time period? Store its 3-letter code in an object called `pessimistic_afinn_country`. 
   - How did the lexicon change your analysis? (We will not grade you for this.)

In [0]:
# ✏️ Calculate net sentiment (Bing)


# ✏️ Calculate net sentiment (Afinn)


# ✏️ Create new data-frame which joins the two net sentiment indices

In [0]:
# ✏️ bing sentiment plot

In [0]:
# ✏️ afinn sentiment plot

## Question 4: Topic Modelling (12 points)

1. Create a document-term matrix - do NOT filter for frequency at this stage. This should take 3 lines of code.  (1 point)
2. Run LDA to obtain **5 topics** for this matrix. 
    - What function would you use to ensure the reproducibility of your LDA? Store the function name in an object called `my_function`. For example, if the code you would run is `make_reproducible()`, store `my_function <- "make_reproducible"`(1 point)
    - Use the function to set your seed as 123. 
    - Visualize the top 10 words in each topic. Store the top word for each topic in a vector called `top_words_5`. What do you notice? (2 points)
    	- For example, if the top word in topics 1-5 is "cat", "dog", "cat", "peac", "fish" respectively, then store your answer as `top_words_5 <- c("cat","dog","cat","peac","fish")` 
3. Now re-run step 1, this time filtering for frequency, using the same thresholds as the lecture (1 point)
4. Re-run LDA to obtain **3 topics** for this matrix. Visualize the top 10 words in each topic. Store the top words for each topic in a vector called `top_words_3` (2 points)
    - For the purposes of this practice set, re-run the code that sets your seed as 123 right before you re-run your LDA
5. Recall `pessimistic_bing_country`. Analyze the topic evolution over time for this country. Which topic saw the biggest increase in prevalence in this country's speeches? Store the topic number in an object called `pessimistic_country_topic`. (2 points)
6. Create a crude topic sentiment index. (3 points)
    - Label each document using its most prevalent topic. E.g. if a document has a prevalence of 0.5, 0.3, and 0.2 for topics 1, 2, and 3, label it as topic 1.
    - The previous step gives you a data-frame of documents and their corresponding labels. Use this data-frame and `left_join(sentiment_df)`
    - Store your results in a data-frame called `topic_sentiments` with 4 columns: `country`, `year`, `topic`, and `sentiment`. **For sentiment, use the afinn lexicon**
    - Visualize the mean sentiment of topics over time. Use the `x`, `y`, and `color` aesthetics in `ggplot`. 
    	- If your plot is displaying in a confusing way, you may try to troubleshoot by using `as.factor(topic)` in your aesthetic.
    - Which topic saw the biggest drop in sentiment over time? Store the topic number in an object called `pessimistic_topic`.
    - What are the disadvantages of this method for analyzing drivers of climate sentiment over time? (We will not grade you on this.)

In [0]:
# ✏️ 1st try at creating DTM

In [0]:
# ✏️ what function do you use to ensure reproducibility?

In [0]:
# ✏️ 1st try at LDA

In [0]:
# ✏️ filter for frequency

In [0]:
# ✏️ re-run LDA

In [0]:
# ✏️ analyze topic evolution

In [0]:
# ✏️ create crude topic sentiment index

---

## How to submit

When you've finished the problem set:

1. In the Colab menu bar, click **File → Save a copy in GitHub**.
2. In the dialog:
   - **Repository**: choose the repository Classroom50 created for you when you accepted this assignment (it will have your GitHub username in the name).
   - **Branch**: `main`.
   - **File path**: leave the filename as-is so it overwrites the starter notebook.
   - **Commit message**: something descriptive, e.g. *"Finished pset"*.
3. Click **OK**. Colab will push your completed notebook back to your assignment repository.

You can save to GitHub as often as you like — each save is just another commit. Your most recent commit before the due date is what gets graded.
